# Prediccion de Churn de Clientes

## Prediccion de Churn: contexto del caso

**Tipo de modelo:** Clasificacion binaria con probabilidades
**Variable objetivo:** churn_90d — P(cancelacion en 90 dias)

**Lo que aprenderas:**
1. Por que la variable objetivo es una probabilidad, no una categoria
2. Train/test split estratificado para clases desbalanceadas
3. Por que accuracy es la metrica equivocada para churn
4. Como elegir el umbral de decision segun el coste del error
5. Como explicar al negocio por que el modelo marco un cliente como en riesgo

---

## Contexto del caso de negocio

| | |
|---|---|
| **Empresa** | empresa — área de ventas y gestión de clientes |
| **Problema de negocio** | Identificar qué clientes tienen alta probabilidad de cancelar su contrato en los próximos 90 días para intervenir antes de que se vayan |
| **Datos disponibles** | 400 registros de clientes: sector, tamaño, antigüedad, número de incidencias, NPS, uso del servicio, importe de contrato y variable objetivo churn_90d |
| **Técnica aplicada** | Clasificación supervisada con GradientBoostingClassifier (modelo principal) y RandomForestClassifier (comparación); umbral de decisión ajustable para priorizar recall sobre precisión |
| **Salida del modelo** | Probabilidad de churn por cliente (0-1) y clasificación binaria según el umbral elegido |
| **Valor operativo** | Permite al equipo comercial ordenar la cartera por riesgo y activar retenciones proactivas solo donde la probabilidad supera el umbral operativo definido |

In [ ]:
import os, sys
from pathlib import Path

# Configuracion de entorno: ajusta CWD y descarga datos segun el entorno de ejecucion
_BASE_URL = "https://raw.githubusercontent.com/amador2001/ia-datos/main/"
_CSVS = ["clientes_empresa.csv"]

if "google.colab" in sys.modules:
    import urllib.request
    Path("datos").mkdir(exist_ok=True)
    for _csv in _CSVS:
        _dest = Path("datos") / _csv
        if not _dest.exists():
            urllib.request.urlretrieve(_BASE_URL + _csv, str(_dest))
            print(f"Descargado: {_csv}")
elif "__vsc_ipynb_file__" in dir():
    os.chdir(Path(__vsc_ipynb_file__).parent)
elif not Path("datos").exists():
    for _p in [Path("Jupyter_notebooks"), Path("../Jupyter_notebooks")]:
        if (_p / "datos").exists():
            os.chdir(_p)
            break

print(f"Entorno listo. CWD: {os.getcwd()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve,
                              precision_recall_curve, f1_score)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

In [ ]:
# Carga del dataset desde CSV
df = pd.read_csv("datos/clientes_empresa.csv")

print("Primeras filas del dataset de clientes:")
display(df.head(10))

print("\nEstadisticas descriptivas:")
display(df.describe().round(3))

print("\nInformacion del dataset:")
df.info()

# Distribucion de la variable objetivo (churn)
# Es importante conocer el desbalance antes de entrenar
print(f"\nDistribucion de churn_90d:")
print(df["churn_90d"].value_counts())
print(f"Tasa de churn: {df['churn_90d'].mean():.1%}")
print("\nNota: con clases desbalanceadas, accuracy no es una metrica util.")
print("Un modelo que predice siempre '0' tendria alta accuracy pero seria inutil.")

### Variables del dataset

| Variable | Tipo | Descripcion |
|---|---|---|
| cliente_id | str | Identificador unico del cliente |
| meses_contrato | int | Antiguedad del contrato en meses |
| tickets_abiertos_90d | int | Incidencias de soporte en los ultimos 90 dias |
| uso_modulos_pct | float | Porcentaje de modulos contratados que usa activamente [0-1] |
| nps_score | int | Net Promoter Score del cliente [-100, +100] |
| retrasos_pago | int | Numero de pagos con retraso en el historico |
| tam_empresa | str | Tamano de la empresa (pequena/mediana/grande) |
| cambios_responsable | int | Veces que cambio el responsable de cuenta del cliente |
| dias_desde_ultimo_contacto | int | Dias desde el ultimo contacto comercial |
| **churn_90d** | int | **Variable objetivo**: 1=cancelo en 90 dias, 0=renovo |

> **Desbalance de clases**: es habitual que la tasa de churn sea baja (10-25%).
> Un modelo que siempre predice "no churn" tendria alta accuracy pero valor cero.
> Por eso se usa AUC-ROC y F1, no accuracy.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

# Variables numericas clave para comparar entre churned y no churned
vars_plot = ["meses_contrato", "uso_modulos_pct", "nps_score",
             "tickets_abiertos_90d", "retrasos_pago", "dias_desde_ultimo_contacto"]

for i, var in enumerate(vars_plot):
    # KDE plot: distribucion de densidad separada por clase
    for label, color in [(0, "steelblue"), (1, "tomato")]:
        subset = df[df["churn_90d"] == label][var]
        subset.plot.kde(ax=axes[i], label=f"churn={label}", color=color, alpha=0.7)
    axes[i].set_title(f"Distribucion: {var}")
    axes[i].set_xlabel(var)
    axes[i].legend()

plt.suptitle("Distribucion de features por clase (churned vs no churned)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Codificar la variable categorica tam_empresa
le = LabelEncoder()
df["tam_empresa_enc"] = le.fit_transform(df["tam_empresa"])  # pequena=2, mediana=1, grande=0

# Features y target
FEATURES = ["meses_contrato", "tickets_abiertos_90d", "uso_modulos_pct",
            "nps_score", "retrasos_pago", "tam_empresa_enc",
            "cambios_responsable", "dias_desde_ultimo_contacto"]
TARGET = "churn_90d"

X = df[FEATURES]
y = df[TARGET]

# Split estratificado: garantiza que ambos splits tienen la misma proporcion de churn
# stratify=y es OBLIGATORIO con clases desbalanceadas
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,       # 25% para test
    random_state=42,
    stratify=y            # mantiene la proporcion de churn en train y test
)

print(f"Filas de entrenamiento: {len(X_train)}")
print(f"Filas de test:          {len(X_test)}")
print(f"Tasa de churn en train: {y_train.mean():.1%}")
print(f"Tasa de churn en test:  {y_test.mean():.1%}  (debe ser similar a train)")

# Normalizar el dataset de entrenamiento y test
# RandomForest no necesita normalizacion, pero LogisticRegression si
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # aprende media y std en train
X_test_scaled  = scaler.transform(X_test)        # aplica la misma transformacion a test

print("\nDataset de entrenamiento normalizado (primeras 3 filas):")
display(pd.DataFrame(X_train_scaled, columns=FEATURES).head(3).round(3))

In [ ]:
# ── Modelo 1: Random Forest (no necesita datos escalados) ────────────────────
rf = RandomForestClassifier(
    n_estimators=100,      # 100 arboles en el ensamble
    max_depth=6,           # limitar profundidad para evitar overfitting
    random_state=42,
    class_weight="balanced"  # penaliza mas los errores en la clase minoritaria (churn=1)
)
rf.fit(X_train, y_train)

# ── Modelo 2: Regresion Logistica (necesita datos escalados) ─────────────────
lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500, random_state=42))
])
lr.fit(X_train, y_train)

# ── Predicciones ─────────────────────────────────────────────────────────────
# predict() devuelve la clase (0 o 1) con umbral por defecto 0.5
y_pred_rf = rf.predict(X_test)
# predict_proba() devuelve la probabilidad de cada clase [prob_0, prob_1]
# Usamos la columna 1 (probabilidad de churn)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("Resultados Random Forest en test:")
print(classification_report(y_test, y_pred_rf, target_names=["Renovo", "Churn"]))
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob_rf):.3f}")

# Mostrar tabla con probabilidades de churn para los primeros 10 clientes del test
tabla_pred = pd.DataFrame({
    "cliente_id":     df.loc[X_test.index[:10], "cliente_id"].values,
    "churn_real":     y_test.values[:10],
    "prob_churn":     y_prob_rf[:10].round(3),
    "pred_umbral_50": y_pred_rf[:10]
})
print("\nPrimeros 10 clientes del test con probabilidad de churn:")
display(tabla_pred)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Curva ROC: muestra el trade-off entre tasa de verdaderos positivos y falsos positivos
fpr, tpr, _ = roc_curve(y_test, y_prob_rf)
auc = roc_auc_score(y_test, y_prob_rf)
axes[0, 0].plot(fpr, tpr, color="steelblue", label=f"Random Forest (AUC={auc:.3f})")
axes[0, 0].plot([0,1],[0,1],"k--", label="Clasificador aleatorio")
axes[0, 0].set_title("Curva ROC")
axes[0, 0].set_xlabel("Tasa de Falsos Positivos")
axes[0, 0].set_ylabel("Tasa de Verdaderos Positivos")
axes[0, 0].legend()

# Matriz de confusion: TP, TN, FP, FN en numeros absolutos
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Pred: Renovo","Pred: Churn"],
            yticklabels=["Real: Renovo","Real: Churn"], ax=axes[0, 1])
axes[0, 1].set_title("Matriz de confusion (umbral=0.5)")

# Feature importance: que variables explican mejor el churn
fi = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)
fi.plot(kind="barh", ax=axes[1, 0], color="steelblue")
axes[1, 0].set_title("Importancia de variables (Random Forest)")
axes[1, 0].set_xlabel("Importancia relativa")

# Efecto del umbral: como cambia precision, recall y F1 al variar el umbral de decision
prec, rec, thresholds = precision_recall_curve(y_test, y_prob_rf)
f1_scores = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
axes[1, 1].plot(thresholds, prec[:-1], label="Precision", color="steelblue")
axes[1, 1].plot(thresholds, rec[:-1],  label="Recall",    color="tomato")
axes[1, 1].plot(thresholds, f1_scores, label="F1",        color="green")
axes[1, 1].axvline(0.5, color="gray", linestyle="--", label="Umbral default 0.5")
axes[1, 1].set_title("Efecto del umbral en metricas")
axes[1, 1].set_xlabel("Umbral de decision")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Prediccion para un cliente nuevo ─────────────────────────────────────────
# Escenario: cliente con senales de riesgo
cliente_nuevo = pd.DataFrame([{
    "meses_contrato":              8,     # cliente reciente (mayor riesgo)
    "tickets_abiertos_90d":        4,     # muchas incidencias recientes
    "uso_modulos_pct":             0.35,  # usa menos de la mitad de los modulos
    "nps_score":                  -5,     # NPS negativo
    "retrasos_pago":               2,     # dos retrasos en el historial
    "tam_empresa_enc":             le.transform(["pequena"])[0],
    "cambios_responsable":         1,     # ha cambiado de responsable
    "dias_desde_ultimo_contacto": 45      # sin contacto comercial en 45 dias
}])

# predict_proba devuelve [prob_renovar, prob_churn]
prob_churn = rf.predict_proba(cliente_nuevo)[0][1]
decision   = rf.predict(cliente_nuevo)[0]

print(f"Probabilidad de churn: {prob_churn:.1%}")
print(f"Decision (umbral 0.5): {'CHURN' if decision == 1 else 'RENOVA'}")
print(f"\nInterpretacion:")
if prob_churn > 0.6:
    print("  ACCION URGENTE: contactar en 48h con oferta de retencion")
elif prob_churn > 0.4:
    print("  SEGUIMIENTO: incluir en campana de nurturing proactivo")
else:
    print("  SIN RIESGO INMEDIATO: mantener ciclo de contacto normal")

### Cuando NO usar este modelo

| Situacion | Por que no funciona |
|---|---|
| Menos de 200 clientes con historico de churn | Sin suficientes ejemplos de la clase minoritaria, el modelo no aprende el patron |
| Churn causado por factores externos | Crisis economica, competidor nuevo: el modelo no los conoce |
| Umbral de decision fijo sin revisar | El umbral optimo cambia segun el coste de retener vs perder un cliente |
| Sin sistema de accion asociado | Predecir sin actuar no genera valor: el modelo debe conectar con un flujo de retencion |

> **Regla de negocio sobre el umbral**: si retener un cliente cuesta 200 EUR y perderlo cuesta 2000 EUR,
> el umbral optimo no es 0.5 sino aproximadamente 0.10 (actuar con cualquier probabilidad > 10%).